Data cleansing and Preprocessing

#   1) Remove all rows with NULL values
#   2) Remove sequences shorter than (mean - std) sequence length
#   3) Create an 80/20 split at SEQUENCE level (no leakage)
#   4) Standardize features (zero mean, unit variance) for MLP-ready inputs

In [1]:
# ==============================
# Preprocessing Imports
# ==============================

# File and path handling
from pathlib import Path
import json
import pickle

# Data manipulation
import pandas as pd
import numpy as np

# Utilities
from typing import List, Dict, Any, Union

# Machine learning tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# ==============================
# Configuration Settings
# ==============================

# Random seed for reproducibility
RANDOM_SEED = 42

# Column name that identifies each sequence
SEQ_COL = "sequence_id"

In [3]:
# ==============================
# Columns to Exclude from Features
# ==============================

# Target labels used only during training
LABEL_COLS = ["gesture", "behavior"]

# Metadata columns that should not be used as model features
META_COLS = [
    "row_id",
    "sequence_type",
    "sequence_counter",
    "subject",
    "orientation",
    "phase"
]

In [4]:
# ==============================
# Sensor Feature Prefixes
# ==============================

# All sensor measurements in the dataset begin with these prefixes.
# They help us automatically identify the numerical sensor features
# described in the paper.
SENSOR_PREFIXES = (
    "acc_",   # accelerometer
    "rot_",   # rotation / gyroscope
    "thm_",   # thermal sensors
    "tof_"    # time-of-flight sensors
)

In [5]:
# ==============================
# Input File Paths
# ==============================

# These files are generated during the data ingestion step.
# They contain the raw sensor sequences for training and testing.
IN_TRAIN_PATH = Path("../data/processed/cmi_sensor_data/train_raw.csv")
IN_TEST_PATH  = Path("../data/processed/cmi_sensor_data/test_raw.csv")

In [6]:
# ==============================
# Output Folder Setup
# ==============================

# All cleaned and processed files will be saved in the same
# processed sensor data directory used by the ingestion step.
OUT_DIR = Path("../data/processed/cmi_sensor_data")

# Create the folder if it does not already exist
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
# ==============================
# Output File Paths
# ==============================

# Cleaned datasets (after NULL removal and sequence filtering)
OUT_TRAIN_CLEAN = OUT_DIR / "train_clean.csv"
OUT_TEST_CLEAN  = OUT_DIR / "test_clean.csv"

# Saved list of sensor feature column names
OUT_FEATURES_JSON = OUT_DIR / "feature_cols.json"

# Sequence-level 80/20 split information
OUT_SPLIT_JSON = OUT_DIR / "split_sequence_ids.json"

# Scaler object (used only for MLP models)
OUT_SCALER_PKL = OUT_DIR / "mlp_standard_scaler.pkl"

OUT_SCALER_PARAMS = OUT_DIR / "mlp_scaler_params.csv"   

# Scaled datasets prepared for MLP training
OUT_TRAIN_SCALED = OUT_DIR / "train_mlp_scaled.csv"
OUT_VAL_SCALED   = OUT_DIR / "val_mlp_scaled.csv"
OUT_TEST_SCALED  = OUT_DIR / "test_mlp_scaled.csv"

# Preprocessing summary report
OUT_REPORT_JSON = OUT_DIR / "preprocessing_report.json"

In [8]:
# ==============================
# Load Raw Train and Test Data
# ==============================

# Check that input files exist before loading
if not IN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Train file not found: {IN_TRAIN_PATH}")

if not IN_TEST_PATH.exists():
    raise FileNotFoundError(f"Test file not found: {IN_TEST_PATH}")

# Read CSV files
train_raw = pd.read_csv(IN_TRAIN_PATH)
test_raw = pd.read_csv(IN_TEST_PATH)

print("Train data shape:", train_raw.shape)
print("Test data shape:", test_raw.shape)

Train data shape: (574945, 341)
Test data shape: (107, 336)


In [9]:
# ==========================================
# Verify required columns exist in the data
# ==========================================

# Make sure sequence_id column exists in both train and test
for name, df in [("train", train_raw), ("test", test_raw)]:
    if SEQ_COL not in df.columns:
        raise KeyError(f"{SEQ_COL} column missing in {name} dataset")

# Labels should exist only in train data
for label in LABEL_COLS:
    if label not in train_raw.columns:
        raise KeyError(f"Missing label column in train data: {label}")

print("Required column check passed")

Required column check passed


In [10]:
# ==========================================
# Identify sensor feature columns
# ==========================================

# Sensor features are columns starting with known prefixes
train_sensor_cols = [c for c in train_raw.columns if c.startswith(SENSOR_PREFIXES)]
test_sensor_cols  = [c for c in test_raw.columns if c.startswith(SENSOR_PREFIXES)]

# Ensure train and test have the same sensor features
if set(train_sensor_cols) != set(test_sensor_cols):
    raise RuntimeError("Train and test sensor columns do not match")

# Final ordered feature list
FEATURE_COLS = sorted(train_sensor_cols)

# Save feature column names for later stages (models)
with open(OUT_FEATURES_JSON, "w") as f:
    json.dump({"feature_cols": FEATURE_COLS}, f, indent=2)

print(f"Total sensor features: {len(FEATURE_COLS)}")

Total sensor features: 332


In [11]:
# ==========================================
# Step 1 — Remove rows containing NULL values
# Paper: "removed all rows with NULL values"
# ==========================================

print("Step 1: Removing rows with missing values...")

# Record original dataset sizes
train_rows_before = len(train_raw)
test_rows_before  = len(test_raw)

# Count rows that contain at least one NULL
train_null_count = train_raw.isna().any(axis=1).sum()
test_null_count  = test_raw.isna().any(axis=1).sum()

# Drop all rows with NULL values
train_no_null = train_raw.dropna().copy()
test_no_null  = test_raw.dropna().copy()

# Record sizes after removal
train_rows_after = len(train_no_null)
test_rows_after  = len(test_no_null)

print(f"Train rows removed: {train_null_count}")
print(f"Test rows removed: {test_null_count}")

Step 1: Removing rows with missing values...
Train rows removed: 38640
Test rows removed: 0


In [12]:
#   Step 2: Remove Short Sequences
#   Keep sequences with length >= (mean - std)
print("Filtering short sequences...")
train_seq_lengths = train_no_null.groupby(SEQ_COL).size()
mean_len = train_seq_lengths.mean()
std_len = train_seq_lengths.std(ddof=0)
threshold = mean_len - std_len
# Identify valid sequences (train and test)
valid_train_seq_ids = train_seq_lengths[train_seq_lengths >= threshold].index
train_clean = train_no_null[train_no_null[SEQ_COL].isin(valid_train_seq_ids)].copy()

test_seq_lengths = test_no_null.groupby(SEQ_COL).size()
valid_test_seq_ids = test_seq_lengths[test_seq_lengths >= threshold].index
test_clean = test_no_null[test_no_null[SEQ_COL].isin(valid_test_seq_ids)].copy()
# Save cleaned (unscaled) datasets
train_clean.to_csv(OUT_TRAIN_CLEAN, index=False)
test_clean.to_csv(OUT_TEST_CLEAN, index=False)

Filtering short sequences...


In [13]:
# ============================================================
# Step 3: Create 80/20 train–validation split at SEQUENCE level
# Requirement from paper:
#   - Split must happen AFTER preprocessing
#   - Split must be sequence-based (not row-based)
#   - No sequence should appear in both sets
# ============================================================

print("Step 3: Creating sequence-level 80/20 train–validation split...")

# ------------------------------------------------------------
# Get list of unique sequence IDs from cleaned training data
# ------------------------------------------------------------
all_sequence_ids = train_clean[SEQ_COL].drop_duplicates().values

# ------------------------------------------------------------
# Try stratified split based on gesture label (if possible)
# This keeps label distribution similar in train/val
# ------------------------------------------------------------
sequence_labels = train_clean.groupby(SEQ_COL)["gesture"].first()

# Align labels with sequence order
labels_for_split = sequence_labels.loc[all_sequence_ids]

# Check if stratification is possible
# (each class must appear at least twice)
can_stratify = labels_for_split.value_counts().min() >= 2

# ------------------------------------------------------------
# Perform 80/20 split at SEQUENCE level
# ------------------------------------------------------------
train_seq_ids, val_seq_ids = train_test_split(
    all_sequence_ids,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=labels_for_split if can_stratify else None
)

print("Train sequences:", len(train_seq_ids))
print("Validation sequences:", len(val_seq_ids))
print("Stratified split used:", can_stratify)

Step 3: Creating sequence-level 80/20 train–validation split...
Train sequences: 6070
Validation sequences: 1518
Stratified split used: True


In [14]:
# Save the train/validation sequence split IDs so results are reproducible
print("Saving sequence split information...")

split_info = {
    "random_seed": RANDOM_SEED,      # ensures same split every run
    "test_size": 0.2,                # 80/20 split
    "stratified": bool(can_stratify),# whether stratified split was used
    "train_seq_ids": train_seq_ids.tolist(),
    "val_seq_ids": val_seq_ids.tolist(),
}

# Write to JSON file
with open(OUT_SPLIT_JSON, "w", encoding="utf-8") as f:
    json.dump(split_info, f, indent=2)

print(f"Sequence split saved → {OUT_SPLIT_JSON}")

Saving sequence split information...
Sequence split saved → ..\data\processed\cmi_sensor_data\split_sequence_ids.json


In [15]:
# Create actual train and validation datasets using the sequence split
print("Creating train and validation datasets from sequence split...")

# Select rows belonging to train sequences
train_split = train_clean[train_clean[SEQ_COL].isin(train_seq_ids)].copy()

# Select rows belonging to validation sequences
val_split = train_clean[train_clean[SEQ_COL].isin(val_seq_ids)].copy()

print(f"Train split rows: {len(train_split)}")
print(f"Validation split rows: {len(val_split)}")

Creating train and validation datasets from sequence split...
Train split rows: 427882
Validation split rows: 108233


In [16]:
# Step 4: Standardize sensor features for MLP models
# Fit scaler on TRAIN split only, then apply to VAL and TEST

print("Fitting StandardScaler on training data...")

# Initialize scaler
mlp_scaler = StandardScaler()

# Extract feature matrices
train_features = train_split[FEATURE_COLS].astype(float)
val_features = val_split[FEATURE_COLS].astype(float)
test_features = test_clean[FEATURE_COLS].astype(float)

# Fit scaler ONLY on training data
train_scaled = mlp_scaler.fit_transform(train_features)

# Apply same scaler to validation and test
val_scaled = mlp_scaler.transform(val_features)
test_scaled = mlp_scaler.transform(test_features)

print("Scaling complete.")
print("Train scaled shape:", train_scaled.shape)
print("Val scaled shape:", val_scaled.shape)
print("Test scaled shape:", test_scaled.shape)

Fitting StandardScaler on training data...
Scaling complete.
Train scaled shape: (427882, 332)
Val scaled shape: (108233, 332)
Test scaled shape: (107, 332)


In [17]:
# Step 5: Save fitted scaler for reproducibility

print("Saving MLP scaler...")

scaler_artifact = {
    "scaler": mlp_scaler,
    "feature_columns": FEATURE_COLS,
    "random_seed": RANDOM_SEED,
    "sequence_length_threshold": float(threshold)
}

with open(OUT_SCALER_PKL, "wb") as f:
    pickle.dump(scaler_artifact, f)

print(f"Scaler saved at: {OUT_SCALER_PKL}")


# ALSO save human-readable scaler parameters (required by report)
print("Saving scaler parameters CSV...")

scaler_params = pd.DataFrame({
    "feature": FEATURE_COLS,
    "mean": mlp_scaler.mean_,
    "std": mlp_scaler.scale_
})

scaler_params.to_csv(OUT_SCALER_PARAMS, index=False)

print(f"Scaler parameters saved at: {OUT_SCALER_PARAMS}")

Saving MLP scaler...
Scaler saved at: ..\data\processed\cmi_sensor_data\mlp_standard_scaler.pkl
Saving scaler parameters CSV...
Scaler parameters saved at: ..\data\processed\cmi_sensor_data\mlp_scaler_params.csv


In [18]:
# Step 6: Save MLP-scaled datasets (train / validation / test)

print("Saving scaled datasets for MLP models...")

# Create copies so original cleaned data remains unchanged
train_mlp_scaled = train_split.copy()
val_mlp_scaled = val_split.copy()
test_mlp_scaled = test_clean.copy()

# Replace only sensor feature columns with scaled values
train_mlp_scaled.loc[:, FEATURE_COLS] = train_scaled
val_mlp_scaled.loc[:, FEATURE_COLS] = val_scaled
test_mlp_scaled.loc[:, FEATURE_COLS] = test_scaled

# Save scaled datasets
train_mlp_scaled.to_csv(OUT_TRAIN_SCALED, index=False)
val_mlp_scaled.to_csv(OUT_VAL_SCALED, index=False)
test_mlp_scaled.to_csv(OUT_TEST_SCALED, index=False)

print("Scaled datasets saved successfully:")
print("  Train:", OUT_TRAIN_SCALED)
print("  Validation:", OUT_VAL_SCALED)
print("  Test:", OUT_TEST_SCALED)

Saving scaled datasets for MLP models...
Scaled datasets saved successfully:
  Train: ..\data\processed\cmi_sensor_data\train_mlp_scaled.csv
  Validation: ..\data\processed\cmi_sensor_data\val_mlp_scaled.csv
  Test: ..\data\processed\cmi_sensor_data\test_mlp_scaled.csv


In [19]:
# Step 7: Generate preprocessing report
# -------------------------------------
print("Creating preprocessing report...")

# Helper: convert numpy/pandas types → JSON-safe Python types
def convert_to_json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value

# Recursive conversion for dict/list structures
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(convert_to_json_safe(v)) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(convert_to_json_safe(v)) for v in obj]
    return convert_to_json_safe(obj)

# Build report dictionary
report = {
    "inputs": {
        "train_raw": str(IN_TRAIN_PATH),
        "test_raw": str(IN_TEST_PATH)
    },
    "outputs": {
        "train_clean_unscaled": str(OUT_TRAIN_CLEAN),
        "test_clean_unscaled": str(OUT_TEST_CLEAN),
        "feature_cols_json": str(OUT_FEATURES_JSON),
        "split_sequence_ids_json": str(OUT_SPLIT_JSON),
        "train_mlp_scaled": str(OUT_TRAIN_SCALED),
        "val_mlp_scaled": str(OUT_VAL_SCALED),
        "test_mlp_scaled": str(OUT_TEST_SCALED),
        "mlp_scaler_pkl": str(OUT_SCALER_PKL),
        "mlp_scaler_params_csv": str(OUT_SCALER_PARAMS),
    },
  "null_removal": {
    "train_rows_before": train_rows_before,
    "train_rows_with_any_null": train_null_count,
    "train_rows_after_dropna": train_rows_after,
    "train_pct_removed": (train_rows_before - train_rows_after) / max(1, train_rows_before) * 100.0,

    "test_rows_before": test_rows_before,
    "test_rows_with_any_null": test_null_count,
    "test_rows_after_dropna": test_rows_after,
    "test_pct_removed": (test_rows_before - test_rows_after) / max(1, test_rows_before) * 100.0,
},
    "sequence_filter": {
        "mean_sequence_length": mean_len,
        "std_sequence_length": std_len,
        "threshold_mean_minus_std": threshold,
        "train_sequences_before": train_no_null[SEQ_COL].nunique(),
        "train_sequences_after": train_clean[SEQ_COL].nunique(),
        "test_sequences_before": test_no_null[SEQ_COL].nunique(),
        "test_sequences_after": test_clean[SEQ_COL].nunique(),
    },
    "features": {
        "sensor_prefixes": list(SENSOR_PREFIXES),
        "num_sensor_features": len(FEATURE_COLS),
        "example_features": FEATURE_COLS[:20],
    },
    "split": {
        "sequence_level_split": True,
        "train_sequences": len(train_seq_ids),
        "val_sequences": len(val_seq_ids),
        "random_seed": RANDOM_SEED,
        "test_size": 0.2,
        "stratified": bool(can_stratify),
    }
}

# Save report to JSON file
try:
    with open(OUT_REPORT_JSON, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(report), f, indent=2)
    print("Preprocessing report saved at:")
    print(OUT_REPORT_JSON)

except Exception as e:
    print("Error saving preprocessing report:", e)

Creating preprocessing report...
Preprocessing report saved at:
..\data\processed\cmi_sensor_data\preprocessing_report.json


In [20]:
# ---------------------------------------------------
# Final Step: Show Summary of Preprocessing Outputs
# ---------------------------------------------------

print("\n" + "=" * 50)
print(" Preprocessing Pipeline Completed Successfully!")
print("=" * 50)

print("\n📁 Saved Files Summary:")

print(f"\n1️⃣ Clean (Unscaled) Data Files:")
print(f"   - Training Data  : {OUT_TRAIN_CLEAN}")
print(f"   - Testing Data   : {OUT_TEST_CLEAN}")

print(f"\n2️⃣ Dataset Split Information:")
print(f"   - Split IDs File : {OUT_SPLIT_JSON}")

print(f"\n3️⃣ Feature Information:")
print(f"   - Feature List   : {OUT_FEATURES_JSON}")

print(f"\n4️⃣ Scaled Data (Ready for MLP Model):")
print(f"   - Train Scaled   : {OUT_TRAIN_SCALED}")
print(f"   - Validation     : {OUT_VAL_SCALED}")
print(f"   - Test Scaled    : {OUT_TEST_SCALED}")

print(f"\n5️⃣ Scaler Files:")
print(f"   - Saved Scaler   : {OUT_SCALER_PKL}")
print(f"   - Scaler Params  : {OUT_SCALER_PARAMS}")

print(f"\n6️⃣ Preprocessing Report:")
print(f"   - Report File    : {OUT_REPORT_JSON}")

print("\n✅ All preprocessing steps completed successfully!")
print("You can now proceed to model training.")
print("=" * 50)



 Preprocessing Pipeline Completed Successfully!

📁 Saved Files Summary:

1️⃣ Clean (Unscaled) Data Files:
   - Training Data  : ..\data\processed\cmi_sensor_data\train_clean.csv
   - Testing Data   : ..\data\processed\cmi_sensor_data\test_clean.csv

2️⃣ Dataset Split Information:
   - Split IDs File : ..\data\processed\cmi_sensor_data\split_sequence_ids.json

3️⃣ Feature Information:
   - Feature List   : ..\data\processed\cmi_sensor_data\feature_cols.json

4️⃣ Scaled Data (Ready for MLP Model):
   - Train Scaled   : ..\data\processed\cmi_sensor_data\train_mlp_scaled.csv
   - Validation     : ..\data\processed\cmi_sensor_data\val_mlp_scaled.csv
   - Test Scaled    : ..\data\processed\cmi_sensor_data\test_mlp_scaled.csv

5️⃣ Scaler Files:
   - Saved Scaler   : ..\data\processed\cmi_sensor_data\mlp_standard_scaler.pkl
   - Scaler Params  : ..\data\processed\cmi_sensor_data\mlp_scaler_params.csv

6️⃣ Preprocessing Report:
   - Report File    : ..\data\processed\cmi_sensor_data\preprocess

In [21]:
# ---------------------------------------------------
# Step: Verify All Preprocessing Output Files Exist
# ---------------------------------------------------

from pathlib import Path

print("\nChecking if all preprocessing output files were created...")

# Define the processed data directory
output_directory = Path("../data/processed/cmi_sensor_data")

# List of files we expect after preprocessing
expected_files = [
    "train_clean.csv",
    "test_clean.csv",
    "feature_cols.json",
    "split_sequence_ids.json",
    "train_mlp_scaled.csv",
    "val_mlp_scaled.csv",
    "test_mlp_scaled.csv",
    "mlp_standard_scaler.pkl",
    "preprocessing_report.json",
]

# Check which files are missing
missing_files = [
    file_name for file_name in expected_files
    if not (output_directory / file_name).exists()
]

# Display results
if len(missing_files) > 0:
    print("\n❌ The following files are missing:")
    for file in missing_files:
        print(f"   - {file}")
    
    # Stop execution if something is wrong
    raise FileNotFoundError("Some preprocessing output files are missing.")

else:
    print("\n✅ All expected preprocessing files are present.")
    print("Your preprocessing pipeline ran successfully!")



Checking if all preprocessing output files were created...

✅ All expected preprocessing files are present.
Your preprocessing pipeline ran successfully!


In [22]:
# ---------------------------------------------------
# Step: Verify No Missing (NULL) Values Remain
# ---------------------------------------------------

import pandas as pd
from pathlib import Path

print("\nChecking for remaining NULL values in cleaned datasets...")

# Define processed data directory (in case not already defined)
out_dir = Path("../data/processed/cmi_sensor_data")

# Load cleaned datasets
train_clean = pd.read_csv(out_dir / "train_clean.csv")
test_clean  = pd.read_csv(out_dir / "test_clean.csv")

# Check for missing values
train_has_nulls = train_clean.isna().any().any()
test_has_nulls  = test_clean.isna().any().any()

if train_has_nulls or test_has_nulls:
    print("\n❌ Missing values detected!")

    if train_has_nulls:
        print("   - train_clean.csv still contains NULL values.")
    if test_has_nulls:
        print("   - test_clean.csv still contains NULL values.")

    raise ValueError("Preprocessing failed: NULL values remain in cleaned datasets.")

else:
    print("\n✅ No NULL values found in train_clean.csv and test_clean.csv.")
    print("Data cleaning step completed successfully!")



Checking for remaining NULL values in cleaned datasets...

✅ No NULL values found in train_clean.csv and test_clean.csv.
Data cleaning step completed successfully!


In [23]:
# ---------------------------------------------------
# Step: Verify Sequence Length Filtering Rule
# ---------------------------------------------------

import numpy as np

print("\nChecking sequence length filtering rule...")

SEQ_COL = "sequence_id"

# Calculate sequence lengths
seq_len = train_clean.groupby(SEQ_COL).size()

# Compute threshold = mean - std
mean_len = float(seq_len.mean())
std_len  = float(seq_len.std(ddof=0))
threshold = mean_len - std_len

# Validate filtering condition
if not (seq_len >= threshold).all():
    raise ValueError("Some sequences do not satisfy the length filtering rule.")

print("\n✅ All train sequences satisfy: length >= (mean - std)")
print(f"Threshold value : {threshold:.4f}")
print(f"Minimum length  : {int(seq_len.min())}")
print("Sequence filtering step verified successfully!")



Checking sequence length filtering rule...

✅ All train sequences satisfy: length >= (mean - std)
Threshold value : 34.8479
Minimum length  : 35
Sequence filtering step verified successfully!


In [24]:
# ---------------------------------------------------
# Step: Verify Sequence-Level Split (No Data Leakage)
# ---------------------------------------------------

import json

print("\nChecking sequence-level train/validation split...")

# Load split file
with open(out_dir / "split_sequence_ids.json", "r") as f:
    split = json.load(f)

# Convert to sets
train_ids = set(split["train_seq_ids"])
val_ids   = set(split["val_seq_ids"])

# 1️⃣ Check no ID overlap
if len(train_ids & val_ids) != 0:
    raise ValueError("Data leakage detected: train and validation IDs overlap.")

# 2️⃣ Filter rows
train_rows = train_clean[train_clean["sequence_id"].isin(train_ids)]
val_rows   = train_clean[train_clean["sequence_id"].isin(val_ids)]

# 3️⃣ Double-check no leakage at row level
if train_rows["sequence_id"].isin(val_ids).any():
    raise ValueError("Leakage detected: validation sequences found in training data.")

if val_rows["sequence_id"].isin(train_ids).any():
    raise ValueError("Leakage detected: training sequences found in validation data.")

print("\n✅ Sequence-level 80/20 split verified.")
print("No overlap detected. No data leakage.")
print(f"Train sequences: {len(train_ids)}")
print(f"Validation sequences: {len(val_ids)}")



Checking sequence-level train/validation split...

✅ Sequence-level 80/20 split verified.
No overlap detected. No data leakage.
Train sequences: 6070
Validation sequences: 1518


In [25]:
# ---------------------------------------------------
# Step: Verify Sensor Feature Columns Consistency
# ---------------------------------------------------

import json

print("\nChecking sensor feature column consistency...")

# Load feature column list
with open(out_dir / "feature_cols.json", "r") as f:
    FEATURE_COLS = json.load(f)["feature_cols"]

# 1️⃣ Check all features exist in train and test
if not set(FEATURE_COLS).issubset(train_clean.columns):
    raise ValueError("Some feature columns are missing in train_clean.csv.")

if not set(FEATURE_COLS).issubset(test_clean.columns):
    raise ValueError("Some feature columns are missing in test_clean.csv.")

# 2️⃣ Extract actual feature sets from both datasets
train_feat = {c for c in train_clean.columns if c in FEATURE_COLS}
test_feat  = {c for c in test_clean.columns if c in FEATURE_COLS}

# 3️⃣ Ensure both datasets have identical feature columns
if train_feat != test_feat:
    raise ValueError("Mismatch between train and test feature columns.")

print("\n✅ Sensor feature columns verified.")
print("Train and test datasets contain identical feature columns.")
print(f"Total feature count: {len(FEATURE_COLS)}")



Checking sensor feature column consistency...

✅ Sensor feature columns verified.
Train and test datasets contain identical feature columns.
Total feature count: 332


In [26]:
# ---------------------------------------------------
# Step: Verify Scaling (Train Mean ≈ 0, Std ≈ 1)
# ---------------------------------------------------

import pandas as pd
import numpy as np

print("\nChecking scaling sanity (mean ≈ 0, std ≈ 1)...")

# Load scaled datasets
train_scaled = pd.read_csv(out_dir / "train_mlp_scaled.csv")
val_scaled   = pd.read_csv(out_dir / "val_mlp_scaled.csv")
test_scaled  = pd.read_csv(out_dir / "test_mlp_scaled.csv")

# Extract feature matrices
X_train = train_scaled[FEATURE_COLS].to_numpy()
X_val   = val_scaled[FEATURE_COLS].to_numpy()
X_test  = test_scaled[FEATURE_COLS].to_numpy()

# Compute statistics on TRAIN only
train_mean = float(X_train.mean())
train_std  = float(X_train.std())

print(f"Train scaled mean: {train_mean:.6f}")
print(f"Train scaled std : {train_std:.6f}")

# Validate scaling (loose bounds)
if abs(train_mean) >= 0.05:
    raise ValueError("Scaling error: train mean is not close to 0.")

if not (0.90 < train_std < 1.10):
    raise ValueError("Scaling error: train std is not close to 1.")

print("\n✅ Scaling verified successfully.")
print("Training data has mean ≈ 0 and standard deviation ≈ 1.")



Checking scaling sanity (mean ≈ 0, std ≈ 1)...
Train scaled mean: 0.000000
Train scaled std : 1.000000

✅ Scaling verified successfully.
Training data has mean ≈ 0 and standard deviation ≈ 1.


In [27]:
# ---------------------------------------------------
# Step: Verify Label Presence (Train Only)
# ---------------------------------------------------

print("\nChecking label column presence...")

# Expected label columns
label_cols = ["gesture", "behavior"]

# 1️⃣ Ensure labels exist in training data
for col in label_cols:
    if col not in train_clean.columns:
        raise ValueError(f"Missing label column in train_clean.csv: {col}")

# 2️⃣ Ensure labels do NOT exist in test data
for col in label_cols:
    if col in test_clean.columns:
        raise ValueError(f"Label leakage detected: {col} found in test_clean.csv")

print("\n✅ Label verification successful.")
print("Labels are present in training data only.")
print("Test data contains no target columns (correct setup).")



Checking label column presence...

✅ Label verification successful.
Labels are present in training data only.
Test data contains no target columns (correct setup).


In [28]:
# Let's check if our preprocessing report file was saved properly

with open(out_dir / "preprocessing_report.json", "r") as f:
    report = json.load(f)

# Make sure important sections exist in the report
if "null_removal" in report and "sequence_filter" in report and "split" in report:
    print("✅ Great! The preprocessing report was created correctly and everything looks good.")
else:
    print("⚠️ Some expected sections are missing from the report.")


✅ Great! The preprocessing report was created correctly and everything looks good.
